# 05 - Add Decision Logic: Relevance And Fallback

Evaluates the relevance-checking and fallback decision layer. By default it uses an offline relevance proxy based on expected document hits for base examples and token overlap for qualitative examples. Optional Tavily execution is disabled by default.


## Learning Goal

Add a decision layer after retrieval: should the system trust local context, or should it fallback to web search? This lab teaches students to evaluate a policy decision, not just a retrieval score.

## Where This Fits

Progression: data sanity -> router eval -> retrieval eval -> cascade eval -> fallback eval -> answer quality -> full benchmark -> ablation.

This lab introduces guardrail-like thinking. The system must decide when retrieved context is good enough and when another source is safer.

## Related AI Evals Concepts

- Types Of Automated Evals: combine deterministic checks, proxy relevance scores, and optional web execution.
- Don't Use Generic Eval Metrics: measure fallback rate, final source, and context relevance directly.
- Traces For Evals: each fallback decision depends on earlier routing and retrieval stages.
- Deploy Evals: fallback behavior resembles a runtime guardrail or recovery path.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

PROJECT_ROOT


In [ ]:
import asyncio
import importlib
import os
from typing import Any

import chromadb
import pandas as pd

import agentic_rag.ingestion as ingestion_module  # noqa: E402
import agentic_rag.retrievers as retrievers_module  # noqa: E402
import agentic_rag.router as router_module  # noqa: E402
from agentic_rag.constants import SourceType  # noqa: E402
from agentic_rag.evaluation import parse_doc_ids, score_retrieval  # noqa: E402

importlib.reload(ingestion_module)
importlib.reload(retrievers_module)
importlib.reload(router_module)

from agentic_rag.ingestion import ensure_chroma_collections  # noqa: E402
from agentic_rag.llm import OpenAITextGenerator  # noqa: E402
from agentic_rag.retrievers import ChromaRetriever  # noqa: E402
from agentic_rag.router import QueryRouter  # noqa: E402
from agentic_rag.settings import Settings  # noqa: E402
from agentic_rag.telemetry import configure_tracing, start_span  # noqa: E402

pd.set_option("display.max_colwidth", 180)


In [ ]:
TRACE_DIR = PROJECT_ROOT / "otel_traces"
TRACE_DIR.mkdir(exist_ok=True)
TRACE_FILE = TRACE_DIR / "05_relevance_and_web_fallback.jsonl"

trace_settings = Settings(
    _env_file=None,
    OTEL_TRACING_ENABLED=True,
    OTEL_TRACES_EXPORTER="file",
    OTEL_TRACES_FILE=TRACE_FILE,
    OTEL_SERVICE_NAME="agentic-rag-notebooks",
)
configure_tracing(trace_settings)

TRACE_FILE


## Load Fallback Test Sets


In [ ]:
base_df = pd.read_csv(PROJECT_ROOT / "datasets/evaluation_dataset.csv")
challenge_df = pd.read_csv(PROJECT_ROOT / "datasets/challenging_router_evaluation_dataset.csv")

if "Query" in base_df.columns:
    base_df = base_df.rename(columns={"Query": "query", "Expected_Source_Type": "expected_source_type"})

base_df["dataset"] = "base"
challenge_df["dataset"] = "challenging"
base_df["expected_source_type"] = base_df["expected_source_type"].astype(str)
challenge_df["expected_source_type"] = challenge_df["expected_source_type"].astype(str)

datasets = pd.concat([base_df, challenge_df], ignore_index=True, sort=False)
datasets[["dataset", "query", "expected_source_type"]].head()


In [ ]:
qna_df = pd.read_csv(PROJECT_ROOT / "datasets/medical_qna_dataset.csv")
device_df = pd.read_csv(PROJECT_ROOT / "datasets/medical_device_manuals_dataset.csv")

settings = Settings(_env_file=None, chroma_path=PROJECT_ROOT / "chroma_db")
client = chromadb.PersistentClient(path=str(settings.chroma_path))

collection_summary = ensure_chroma_collections(client, qna_df, device_df)
collection_summary


## Define Relevance And Fallback Rules


In [ ]:
LABELS = [source.value for source in SourceType]
LOCAL_SOURCES = {SourceType.RETRIEVE_QNA.value, SourceType.RETRIEVE_DEVICE.value}
settings = Settings(_env_file=None, chroma_path=PROJECT_ROOT / "chroma_db")


def source_or_none(value: str) -> SourceType | None:
    try:
        return SourceType(value)
    except ValueError:
        return None


def is_local_source(value: str) -> bool:
    return value in LOCAL_SOURCES


def lexical_score(question: str, answer: str) -> float:
    q_tokens = {token.lower().strip(".,?!:;()[]{}\"'") for token in question.split() if len(token) > 3}
    a_tokens = {token.lower().strip(".,?!:;()[]{}\"'") for token in answer.split() if len(token) > 3}
    if not q_tokens or not a_tokens:
        return 0.0
    return len(q_tokens & a_tokens) / len(q_tokens)


def preview_text(text: str, max_words: int = 50) -> str:
    return " ".join(text.split()[:max_words])

RUN_TAVILY_SEARCH = False

async def fallback_one(row: pd.Series, top_k: int = 3) -> dict[str, Any]:
    query = str(row["query"])
    expected_source = str(row["expected_source_type"])
    expected_ids = parse_doc_ids(row.get("expected_doc_ids", ""))
    router = QueryRouter(mode="heuristic")

    with start_span("notebook.fallback.row", dataset=str(row["dataset"]), query_length=len(query)):
        predicted_source = (await router.route(query)).value
        retrieved_ids: list[str] = []
        context = ""

        if is_local_source(predicted_source):
            retriever = ChromaRetriever(chroma_path=str(settings.chroma_path), top_k=top_k)
            docs = await retriever.retrieve(SourceType(predicted_source), query)
            retrieved_ids = [doc.doc_id for doc in docs]
            context = "\n".join(doc.text for doc in docs)

        if expected_ids:
            retrieval_score = score_retrieval(retrieved_ids, expected_ids)
            context_relevant = retrieval_score.hit_at_k == 1
            relevance_score = retrieval_score.hit_at_k
            relevance_status = "gold_doc_hit_proxy"
        elif context:
            relevance_score = lexical_score(query, context)
            context_relevant = relevance_score >= 0.2
            relevance_status = "lexical_proxy_no_gold"
        else:
            relevance_score = None
            context_relevant = predicted_source == SourceType.WEB_SEARCH.value
            relevance_status = "no_local_context"

        fallback_triggered = is_local_source(predicted_source) and not context_relevant
        final_source = SourceType.WEB_SEARCH.value if fallback_triggered else predicted_source

        web_context_length = None
        if RUN_TAVILY_SEARCH and final_source == SourceType.WEB_SEARCH.value:
            from agentic_rag.web_search import TavilyWebSearcher  # noqa: E402

            web_context = await TavilyWebSearcher(max_results=1).search(query)
            web_context_length = len(web_context)

        return {
            "dataset": row["dataset"],
            "query": query,
            "expected_source_type": expected_source,
            "predicted_source_type": predicted_source,
            "retrieved_doc_ids": "|".join(retrieved_ids),
            "context_relevant": context_relevant,
            "relevance_score": relevance_score,
            "relevance_status": relevance_status,
            "fallback_triggered": fallback_triggered,
            "final_source_type": final_source,
            "web_context_length": web_context_length,
            "top_context_preview": preview_text(context) if context else None,
        }


async def evaluate_fallback(df: pd.DataFrame) -> pd.DataFrame:
    rows = await asyncio.gather(*(fallback_one(row) for _, row in df.iterrows()))
    return pd.DataFrame(rows)


## Measure Fallback On Clear Examples


In [ ]:
base_fallback_results = await evaluate_fallback(base_df)
base_fallback_results.head()


In [ ]:
base_fallback_summary = pd.DataFrame([
    {
        "dataset": "base",
        "examples": len(base_fallback_results),
        "fallback_rate": float(base_fallback_results["fallback_triggered"].mean()),
        "local_context_relevance_rate": float(base_fallback_results.loc[base_fallback_results["predicted_source_type"].isin(LOCAL_SOURCES), "context_relevant"].mean()),
        "web_final_rate": float((base_fallback_results["final_source_type"] == SourceType.WEB_SEARCH.value).mean()),
    }
])
base_fallback_summary


## Inspect Fallback On Hard Examples


In [ ]:
challenge_fallback_results = await evaluate_fallback(challenge_df)
challenge_fallback_results.head()


In [ ]:
challenge_fallback_summary = pd.DataFrame([
    {
        "dataset": "challenging",
        "examples": len(challenge_fallback_results),
        "fallback_rate": float(challenge_fallback_results["fallback_triggered"].mean()),
        "web_final_rate": float((challenge_fallback_results["final_source_type"] == SourceType.WEB_SEARCH.value).mean()),
        "router_accuracy": float((challenge_fallback_results["expected_source_type"] == challenge_fallback_results["predicted_source_type"]).mean()),
    }
])
challenge_fallback_summary


In [ ]:
challenge_fallback_results[["query", "expected_source_type", "predicted_source_type", "context_relevant", "fallback_triggered", "final_source_type", "relevance_status", "top_context_preview"]]


## Pay Attention To

- Fallback is a policy decision: too little fallback can preserve bad context, too much fallback can waste cost and latency.
- Proxy relevance is useful for teaching, but it should not be confused with a perfect human judgment.
- `Web_Search` examples often have different evaluation needs than local retrieval examples.
- Runtime guardrails should be fast, interpretable, and tied to a specific failure mode.


## Optional Advanced Path: LLM Router Evaluation

This section is disabled by default because it makes API calls. Set `RUN_LLM_ROUTER = True` to compare the LLM router against both the base and challenging router datasets. It reads `OPENAI_API_KEY` from the active environment only; it does not load `.env`.


In [ ]:
RUN_LLM_ROUTER = True
LLM_ROUTER_MODEL = os.getenv("OPENAI_ROUTER_MODEL", "gpt-5-nano")
LLM_ROUTER_TIMEOUT = float(os.getenv("OPENAI_TIMEOUT", "60"))

llm_router_results = pd.DataFrame()
llm_router_summary = pd.DataFrame()


def load_router_eval_dataset(path: Path, dataset_name: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    if "Query" in df.columns:
        df = df.rename(columns={"Query": "query", "Expected_Source_Type": "expected_source_type"})
    df = df.copy()
    df["dataset"] = dataset_name
    df["expected_source_type"] = df["expected_source_type"].astype(str)
    return df


async def evaluate_llm_router_dataset(router: QueryRouter, df: pd.DataFrame) -> pd.DataFrame:
    async def route_row(query: str) -> str:
        return (await router.route(query)).value

    predictions = await asyncio.gather(*(route_row(str(query)) for query in df["query"].tolist()))
    results = df.copy()
    results["router_mode"] = "llm"
    results["predicted_source_type"] = predictions
    results["route_correct"] = results["expected_source_type"] == results["predicted_source_type"]
    keep_columns = [
        "dataset",
        "router_mode",
        "query",
        "expected_source_type",
        "predicted_source_type",
        "route_correct",
        "category",
        "rationale",
    ]
    return results[[column for column in keep_columns if column in results.columns]]


def summarize_llm_router_results(results: pd.DataFrame) -> pd.DataFrame:
    if results.empty:
        return pd.DataFrame()
    return results.groupby(["dataset", "router_mode"], dropna=False).agg(
        examples=("query", "count"),
        accuracy=("route_correct", "mean"),
        failures=("route_correct", lambda values: int((~values).sum())),
    ).reset_index()


if RUN_LLM_ROUTER:
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise RuntimeError("OPENAI_API_KEY must be set in the active environment for LLM router evaluation")

    llm_router = QueryRouter(
        generator=OpenAITextGenerator(api_key=api_key, model=LLM_ROUTER_MODEL, timeout=LLM_ROUTER_TIMEOUT),
        mode="llm",
    )
    llm_router_base_df = load_router_eval_dataset(PROJECT_ROOT / "datasets/evaluation_dataset.csv", "base")
    llm_router_challenge_df = load_router_eval_dataset(PROJECT_ROOT / "datasets/challenging_router_evaluation_dataset.csv", "challenging")
    llm_router_results = pd.concat(
        [
            await evaluate_llm_router_dataset(llm_router, llm_router_base_df),
            await evaluate_llm_router_dataset(llm_router, llm_router_challenge_df),
        ],
        ignore_index=True,
    )
    llm_router_summary = summarize_llm_router_results(llm_router_results)

llm_router_summary


In [ ]:
if not llm_router_results.empty:
    display(llm_router_summary)
    display(pd.crosstab(
        [llm_router_results["dataset"], llm_router_results["expected_source_type"]],
        llm_router_results["predicted_source_type"],
        dropna=False,
    ))
    display(llm_router_results.loc[~llm_router_results["route_correct"]])


## Export Fallback Artifacts


In [ ]:
fallback_results = pd.concat([base_fallback_results, challenge_fallback_results], ignore_index=True)
fallback_summary = pd.concat([base_fallback_summary, challenge_fallback_summary], ignore_index=True)

results_path = PROJECT_ROOT / "relevance_fallback_results.csv"
summary_path = PROJECT_ROOT / "relevance_fallback_summary.csv"
fallback_results.to_csv(results_path, index=False)
fallback_summary.to_csv(summary_path, index=False)

results_path, summary_path
